In [1]:
import duckdb
con = duckdb.connect("olist.duckdb")
print("connected")

connected


In [2]:
import numpy as np
import pandas as pd

In [3]:
from scipy.stats import norm
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize, proportions_ztest

In [4]:
# ---------------- ASSUMPTIONS (simulated experiment, not Olist data) ----------------
BASELINE       = 0.030   # assumed conversion rate with current shipping (3.0%)
TREATMENT_TRUE = 0.033   # assumed true rate with free shipping (a 10% relative lift)
ALPHA          = 0.05    # significance level
POWER          = 0.80    # chance of detecting the lift if it is real
SEED           = 42
# -------------------------------------------------------------------------------------

def sample_size(p_control, p_treat):
    h = proportion_effectsize(p_treat, p_control)
    return int(np.ceil(NormalIndPower().solve_power(effect_size=h, alpha=ALPHA, power=POWER,
                                                    ratio=1.0, alternative="two-sided")))

n_per_group = sample_size(BASELINE, TREATMENT_TRUE)
print("Visitors needed per group:", f"{n_per_group:,}")
print("Total visitors needed:", f"{2 * n_per_group:,}")

# How the required sample changes with the size of the lift you want to detect
sens = pd.DataFrame({"relative_lift": [0.05, 0.10, 0.15, 0.20]})
sens["treatment_rate"] = BASELINE * (1 + sens.relative_lift)
sens["visitors_per_group"] = [sample_size(BASELINE, t) for t in sens.treatment_rate]
sens

Visitors needed per group: 53,182
Total visitors needed: 106,364


,relative_lift,treatment_rate,visitors_per_group
0,0.05,0.0315,207909
1,0.10,0.0330,53182
2,0.15,0.0345,24165
3,0.20,0.0360,13887


In [5]:
con.execute("""
CREATE OR REPLACE TABLE ab_threshold AS
WITH t AS (
  SELECT QUANTILE_CONT(item_revenue, 0.75) AS threshold
  FROM order_level
  WHERE order_status = 'delivered' AND item_revenue IS NOT NULL
)
SELECT ROUND(t.threshold, 2) AS threshold,
       COUNT(*) AS delivered_orders,
       SUM(CASE WHEN o.item_revenue >= t.threshold THEN 1 ELSE 0 END) AS eligible_orders,
       ROUND(100.0 * SUM(CASE WHEN o.item_revenue >= t.threshold THEN 1 ELSE 0 END) / COUNT(*), 2) AS eligible_pct,
       ROUND(SUM(CASE WHEN o.item_revenue >= t.threshold THEN o.freight ELSE 0 END), 2) AS freight_absorbed,
       ROUND(AVG(CASE WHEN o.item_revenue >= t.threshold THEN o.freight END), 2) AS avg_freight_eligible,
       ROUND(AVG(CASE WHEN o.item_revenue >= t.threshold THEN o.item_revenue END), 2) AS avg_value_eligible,
       ROUND(SUM(o.item_revenue), 2) AS revenue
FROM order_level o
CROSS JOIN t
WHERE o.order_status = 'delivered' AND o.item_revenue IS NOT NULL
GROUP BY t.threshold
""")

con.execute("SELECT * FROM ab_threshold").df()

,threshold,delivered_orders,eligible_orders,eligible_pct,freight_absorbed,avg_freight_eligible,avg_value_eligible,revenue
0,149.9,96478,24637.0,25.54,856507.9,34.77,337.29,13221498.11


In [6]:
rng = np.random.default_rng(SEED)
control = rng.binomial(1, BASELINE, n_per_group)
treat   = rng.binomial(1, TREATMENT_TRUE, n_per_group)

In [7]:
counts = np.array([treat.sum(), control.sum()])
nobs   = np.array([n_per_group, n_per_group])
z_stat, p_value = proportions_ztest(counts, nobs, alternative="two-sided")

In [8]:
p_t, p_c = counts / nobs
diff = p_t - p_c
se = np.sqrt(p_t * (1 - p_t) / n_per_group + p_c * (1 - p_c) / n_per_group)
ci_low, ci_high = diff - 1.96 * se, diff + 1.96 * se

In [9]:
print(f"Control:   {control.sum():,} orders from {n_per_group:,} visitors = {100 * p_c:.2f}%")
print(f"Treatment: {treat.sum():,} orders from {n_per_group:,} visitors = {100 * p_t:.2f}%")
print(f"Difference: {100 * diff:.2f} percentage points (95% CI {100 * ci_low:.2f} to {100 * ci_high:.2f})")
print(f"Relative lift: {100 * diff / p_c:.1f}%")
print(f"z = {z_stat:.2f}, p-value = {p_value:.4f}")
print("Significant at 5%:", p_value < ALPHA)

Control:   1,566 orders from 53,182 visitors = 2.94%
Treatment: 1,782 orders from 53,182 visitors = 3.35%
Difference: 0.41 percentage points (95% CI 0.20 to 0.62)
Relative lift: 13.8%
z = 3.79, p-value = 0.0001
Significant at 5%: True


In [10]:
def rejection_rate(p_control, p_treat, sims=2000, seed=1):
    rng = np.random.default_rng(seed)
    c = rng.binomial(n_per_group, p_control, sims)
    t = rng.binomial(n_per_group, p_treat, sims)
    pc, pt = c / n_per_group, t / n_per_group
    pooled = (c + t) / (2 * n_per_group)
    se = np.sqrt(pooled * (1 - pooled) * 2 / n_per_group)
    z = (pt - pc) / se
    p = 2 * (1 - norm.cdf(np.abs(z)))
    return float((p < ALPHA).mean())

power_sim = rejection_rate(BASELINE, TREATMENT_TRUE)
false_pos_sim = rejection_rate(BASELINE, BASELINE)

print(f"Real 10% lift detected in {100 * power_sim:.1f}% of 2,000 simulated experiments (target {100 * POWER:.0f}%)")
print(f"False alarms when there is no real change: {100 * false_pos_sim:.1f}% (target {100 * ALPHA:.0f}%)")

Real 10% lift detected in 79.0% of 2,000 simulated experiments (target 80%)
False alarms when there is no real change: 5.5% (target 5%)


In [11]:
t = con.execute("SELECT * FROM ab_threshold").df().iloc[0]

results = pd.DataFrame([{
    "simulated": True,
    "baseline_rate_pct": 100 * BASELINE,
    "assumed_true_treatment_rate_pct": 100 * TREATMENT_TRUE,
    "alpha": ALPHA,
    "target_power": POWER,
    "visitors_per_group": n_per_group,
    "control_rate_pct": round(100 * p_c, 3),
    "treatment_rate_pct": round(100 * p_t, 3),
    "difference_pp": round(100 * diff, 3),
    "ci_low_pp": round(100 * ci_low, 3),
    "ci_high_pp": round(100 * ci_high, 3),
    "relative_lift_pct": round(100 * diff / p_c, 2),
    "z_stat": round(float(z_stat), 3),
    "p_value": round(float(p_value), 5),
    "significant": bool(p_value < ALPHA),
    "power_in_2000_sims_pct": round(100 * power_sim, 1),
    "false_alarm_in_2000_sims_pct": round(100 * false_pos_sim, 1),
}])

results.to_csv("dashboard/ab_test_results.csv", index=False)
con.execute("COPY ab_threshold TO 'dashboard/ab_threshold.csv' (HEADER, DELIMITER ',')")
print("saved")
results.T

saved


,0
simulated,True
baseline_rate_pct,3.0
assumed_true_treatment_rate_pct,3.3
alpha,0.05
target_power,0.8
visitors_per_group,53182
control_rate_pct,2.945
treatment_rate_pct,3.351
difference_pp,0.406
ci_low_pp,0.196


In [12]:
def n0(x):
    return f"{x:,.0f}"

freight_pct_rev = 100 * t.freight_absorbed / t.revenue
freight_per_order = t.freight_absorbed / t.delivered_orders

if p_value < ALPHA:
    outcome = (f"In this simulated run, the free-shipping group converted at {100 * p_t:.2f}% against {100 * p_c:.2f}% for control, "
               f"a difference of {100 * diff:.2f} percentage points (95% confidence interval {100 * ci_low:.2f} to {100 * ci_high:.2f}), "
               f"p = {p_value:.4f}. The result is statistically significant at the 5% level.")
else:
    outcome = (f"In this simulated run, the free-shipping group converted at {100 * p_t:.2f}% against {100 * p_c:.2f}% for control, "
               f"a difference of {100 * diff:.2f} percentage points (95% confidence interval {100 * ci_low:.2f} to {100 * ci_high:.2f}), "
               f"p = {p_value:.4f}. The result is not statistically significant at the 5% level, even though a real lift was built into the simulation. "
               f"With {100 * POWER:.0f}% power this happens in about {100 * (1 - POWER):.0f}% of experiments, which is why one result should never be read in isolation.")

sens_rows = "\n".join(
    f"| {100 * x.relative_lift:.0f}% | {100 * x.treatment_rate:.2f}% | {n0(x.visitors_per_group)} |"
    for x in sens.itertuples()
)

md = f"""# Simulated A/B test: free-shipping threshold

**IMPORTANT: this experiment is simulated.** Olist has no visitor or experiment data. The visitors, conversions and the size of the lift are generated by code from assumptions. The aim is to show the method (hypothesis, sample size, test, decision rule), not to claim a real result about Olist.

## 1. Hypothesis
- **Change:** orders with an item value of {t.threshold:,.2f} or more ship free (the 75th percentile of delivered order values in the data).
- **Null hypothesis:** free shipping above the threshold does not change the visitor-to-order conversion rate.
- **Alternative hypothesis:** free shipping changes the conversion rate (two-sided test).

## 2. Design
- **Unit and split:** visitors are randomly assigned 50/50 to control (current shipping) or treatment (free shipping above the threshold).
- **Primary metric:** conversion rate = orders / visitors.
- **Guardrail metrics:** average order value, freight cost absorbed per order, delivery lateness and review score.
- **Significance level:** {ALPHA}. **Power:** {POWER}.

## 3. Assumptions (not measured)
- Baseline conversion rate: {100 * BASELINE:.1f}%.
- Smallest lift worth detecting: {100 * TREATMENT_TRUE:.1f}% under treatment, a {100 * (TREATMENT_TRUE / BASELINE - 1):.0f}% relative lift.
- These rates are illustrative; Olist's real visitor-level conversion is unknown.

## 4. Sample size
Each group needs **{n0(n_per_group)} visitors** ({n0(2 * n_per_group)} in total) to detect a lift from {100 * BASELINE:.1f}% to {100 * TREATMENT_TRUE:.1f}% with {100 * POWER:.0f}% power at the 5% level.

| Relative lift to detect | Treatment rate | Visitors needed per group |
|---|---|---|
{sens_rows}

Smaller lifts need far more visitors, so the lift worth detecting should be decided before the experiment starts.

## 5. Simulated result
{outcome}

## 6. How reliable the test is
Repeating the experiment 2,000 times in simulation, the test detected the real lift in {100 * power_sim:.1f}% of runs (target {100 * POWER:.0f}%) and raised a false alarm in {100 * false_pos_sim:.1f}% of runs when nothing had changed (target {100 * ALPHA:.0f}%).

## 7. Cost of the change, measured from Olist data
- {t.eligible_pct}% of delivered orders ({n0(t.eligible_orders)} of {n0(t.delivered_orders)}) would qualify for free shipping at the {t.threshold:,.2f} threshold.
- Covering their freight would have cost about {n0(t.freight_absorbed)}, which is {freight_pct_rev:.1f}% of delivered item revenue, or {freight_per_order:.2f} per delivered order.
- Qualifying orders average {t.avg_value_eligible:,.2f} in item value and {t.avg_freight_eligible:,.2f} in freight.
- The data has no product costs, so it cannot show whether extra orders would pay for this freight. That depends on margin.

## 8. Decision rule (set before the test)
Roll out only if (1) conversion is significantly higher at the 5% level, (2) average order value does not fall, and (3) the freight absorbed per order stays within an agreed budget that margin can cover. Otherwise do not roll out, or test a higher threshold.

## 9. Limits
- Everything about visitors and conversions is simulated, so a significant result here is built in by the assumed lift.
- Baseline rate and lift are assumptions.
- A real test would also need to check for novelty effects, seasonality and customers who see both versions.
"""

with open("memo/ab_test.md", "w", encoding="utf-8") as fh:
    fh.write(md)
print("written memo/ab_test.md")

written memo/ab_test.md
